# 01 · Random forest on the raw target (first experiment)

Predicts the full-bloom day of year (DOY) directly. This was the first RF experiment, before the team moved to the anomaly target (notebook 02).

| | |
|---|---|
| Datasets | Mar15 / Mar1 / Feb1 × `large` (82 stations) / `small` (48 stations with a 2025 record) |
| Training window | `all` = every training year of the fold · `30y` = only the last 30 training years |
| Grid | 96 candidates: `n_estimators` 200/500 · `max_depth` 5/10/15/None · `min_samples_leaf` 1/10/30/50 · `max_features` sqrt/0.7/1.0 |
| CV | the five agreed expanding-window folds (validation 2005–2014). Test years 2015–2025 are never used |
| Output | `results/rf/01_raw_target/` |

**Status: kept as a record, not rerun in the clean-up.** The original run stopped after 7 of the 12 conditions
(Mar15 × 4, Mar1-large × 2, Mar1-small-all). **Mar1-small-30y and all four Feb1 conditions were never run**, so the raw target
can only be compared with the anomaly target for 15 March and 1 March. In those runs the anomaly target was better
(Mar15: 3.06 vs 3.14 days, Mar1: 3.56 vs 3.88 days) and every `30y` window was worse than `all`.

A full run is 96 × 5 × 12 = 5,760 fits and takes hours. Set `MAX_CANDIDATES = 2` and `OUTPUT_OVERRIDE` to a scratch folder to test the code.

In [ ]:
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import ParameterGrid

from rf_common import *

MAX_CANDIDATES = None    # None = full grid; e.g. 2 = quick test
OUTPUT_OVERRIDE = None   # None = results/rf/01_raw_target; set a scratch folder for test runs
N_JOBS = -1

OUT = output_dir('01_raw_target', OUTPUT_OVERRIDE)
print('Results go to:', OUT)

## 1. Datasets

In [ ]:
datasets = {
    dataset_name(date, sample): load_sample(date, sample)
    for date in ['mar15', 'mar1', 'feb1']
    for sample in SAMPLES
}

display(pd.DataFrame([
    {
        'dataset': name,
        'rows': len(df),
        'stations': df['station_id'].nunique(),
        'development_rows': (df['split'] == 'development').sum(),
        'test_rows': (df['split'] == 'test').sum(),
        'features': len(feature_columns(df)),
    }
    for name, df in datasets.items()
]))

## 2. Hyperparameter grid and training windows

In [ ]:
param_grid = {
    'n_estimators': [200, 500],
    'max_depth': [5, 10, 15, None],
    'min_samples_leaf': [1, 10, 30, 50],
    'max_features': ['sqrt', 0.7, 1.0],
}
candidates = list(ParameterGrid(param_grid))[:MAX_CANDIDATES]

# all: every training year of the fold; 30y: only the latest 30 training years
training_windows = {'all': None, '30y': 30}

print('Candidates per condition:', len(candidates))
print('Conditions:', len(datasets) * len(training_windows))
print('Model fits:', len(candidates) * 5 * len(datasets) * len(training_windows))

## 3. Grid search

Each condition is saved as soon as it finishes:
`{date}_{sample}_train{window}_grid_search_all_candidates.csv`, `..._all_candidates_fold_results.csv`,
`..._best_candidate_fold_results.csv`, and one row per condition in `best_models.csv`.

In [ ]:
best_models_path = OUT / 'best_models.csv'
comparison_rows = {}
if best_models_path.exists():
    for row in read_csv(best_models_path).to_dict('records'):
        comparison_rows[row['dataset']] = row

for base_name, df in datasets.items():
    features = feature_columns(df)
    date_label, sample = base_name.split('-')
    original_folds = list(cv_folds(df))

    for window_name, window_years in training_windows.items():
        name = f'{base_name}-{window_name}'
        prefix = f'{DATES[date_label]}_{sample}_train{window_name}'

        # Validation rows never change; only the training years are shortened.
        folds = []
        for fold, train, valid in original_folds:
            check_folds(train, valid, f'{name} fold {fold}')
            if window_years is not None:
                train_end = int(train['bloom_year'].max())
                train = train.loc[
                    train['bloom_year'].between(train_end - window_years + 1, train_end)
                ].copy()
            folds.append((fold, train, valid))

        fold_rows = []
        for candidate_id, params in enumerate(candidates):
            for fold, train, valid in folds:
                model = RandomForestRegressor(**params, random_state=RANDOM_STATE, n_jobs=N_JOBS)
                model.fit(train[features], train[TARGET])
                train_scores = evaluate(train[TARGET], model.predict(train[features]))
                valid_scores = evaluate(valid[TARGET], model.predict(valid[features]))
                fold_rows.append({
                    'candidate_id': candidate_id,
                    'fold': fold,
                    'train_MAE': train_scores['MAE'],
                    'valid_MAE': valid_scores['MAE'],
                    'valid_RMSE': valid_scores['RMSE'],
                    'valid_R2': valid_scores['R2'],
                    'valid_hit_rate_3d': valid_scores['hit_rate_3d'],
                })

        fold_results = pd.DataFrame(fold_rows)

        # Equal-weight mean over the five validation folds.
        mean_scores = fold_results.groupby('candidate_id').agg(
            train_MAE=('train_MAE', 'mean'),
            valid_MAE=('valid_MAE', 'mean'),
            valid_MAE_std=('valid_MAE', 'std'),
            valid_RMSE=('valid_RMSE', 'mean'),
            valid_R2=('valid_R2', 'mean'),
            valid_hit_rate_3d=('valid_hit_rate_3d', 'mean'),
        )
        parameter_table = pd.DataFrame(candidates)
        parameter_table.index.name = 'candidate_id'
        gs_results = parameter_table.join(mean_scores).sort_values('valid_MAE', kind='stable')

        best_id = int(gs_results.index[0])
        best_params = candidates[best_id].copy()     # keeps max_depth=None as None
        best_fold_results = (
            fold_results.loc[fold_results['candidate_id'] == best_id]
            .sort_values('fold').reset_index(drop=True)
        )

        save_csv(gs_results, OUT / f'{prefix}_grid_search_all_candidates.csv', index=True)
        save_csv(fold_results, OUT / f'{prefix}_all_candidates_fold_results.csv')
        save_csv(best_fold_results, OUT / f'{prefix}_best_candidate_fold_results.csv')

        best = gs_results.loc[best_id]
        comparison_rows[name] = {
            'dataset': name,
            'base_dataset': base_name,
            'training_window': window_name,
            'features': len(features),
            'best_candidate_id': best_id,
            **best_params,
            **best[['train_MAE', 'valid_MAE', 'valid_MAE_std', 'valid_RMSE',
                    'valid_R2', 'valid_hit_rate_3d']].to_dict(),
        }
        comparison = pd.DataFrame(list(comparison_rows.values()))
        save_csv(comparison, best_models_path)

        print(f'Saved {prefix}: best {best_params}, MAE {best["valid_MAE"]:.4f}', flush=True)

display(comparison.round(3))